# Orquestador Multimodal IRON-SYNC

Entrenamiento del clasificador de movimientos basado en IMU + EMG + ECG.

**13 movimientos**: IDLE, WALK, RUN, JUMP, CROUCH, PUNCH_R/L, KICK_R/L, BLOCK, WAVE, SHOOT, FLY

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parents[3]
sys.path.insert(0, str(ROOT / "IA-IRON-SYNC" / "orquestador"))

import numpy as np
import torch
import matplotlib.pyplot as plt
import seaborn as sns

from src.config import OrquestadorConfig, MOVEMENTS, MOVEMENT_NAMES, NUM_MOVEMENTS
from src.model import create_orquestador
from src.data_generator import generate_dataset
from src.trainer import OrquestadorTrainer

print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
print(f"Movimientos: {NUM_MOVEMENTS}")
for name in MOVEMENT_NAMES:
    m = MOVEMENTS[name]
    print(f"  [{m.label_es:>16}] {m.description}")

## 1. Generar Dataset Sintético

In [ ]:
config = OrquestadorConfig(
    train_samples=50000,
    val_samples=10000,
    test_samples=10000,
    hidden_dim=256,
    num_layers=4,
    dropout=0.3,
    use_attention=True,
    batch_size=128,
    epochs=100,
    learning_rate=1e-3,
    patience=15,
    target_accuracy=0.85,
    target_f1=0.80
)

dataset_dir = Path("dataset")
dataset = generate_dataset(config, dataset_dir)

## 2. Crear Modelo

In [ ]:
model = create_orquestador(config)
print(f"Parámetros: {model.count_parameters():,}")
print(f"Arquitectura:")
print(model)

## 3. Entrenamiento

In [ ]:
output_dir = Path("checkpoints")

trainer = OrquestadorTrainer(config)
loaders = trainer.load_dataset(dataset_dir)

results = trainer.train(loaders["train"], loaders["val"], output_dir)

## 4. Evaluación en Test

In [ ]:
test_metrics = trainer.test(loaders["test"], output_dir)

## 5. Visualización de Resultados

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# Loss
axes[0, 0].plot(results["history"]["train_loss"], label="Train")
axes[0, 0].plot(results["history"]["val_loss"], label="Val")
axes[0, 0].set_title("Loss")
axes[0, 0].legend()
axes[0, 0].grid(True)

# Accuracy
axes[0, 1].plot(results["history"]["train_acc"], label="Train")
axes[0, 1].plot(results["history"]["val_acc"], label="Val")
axes[0, 1].set_title("Accuracy")
axes[0, 1].legend()
axes[0, 1].grid(True)

# F1 Score
axes[1, 0].plot(results["history"]["val_f1"], color="green")
axes[1, 0].axhline(y=config.target_f1, color="r", linestyle="--", label=f"Target {config.target_f1:.0%}")
axes[1, 0].set_title("F1 Score (Val)")
axes[1, 0].legend()
axes[1, 0].grid(True)

# Learning Rate
axes[1, 1].plot(results["history"]["lr"], color="orange")
axes[1, 1].set_title("Learning Rate")
axes[1, 1].set_yscale("log")
axes[1, 1].grid(True)

plt.tight_layout()
plt.savefig("checkpoints/training_curves.png", dpi=150)
plt.show()

In [ ]:
# Matriz de confusión
confusion = np.load("checkpoints/confusion_matrix.npy")

plt.figure(figsize=(14, 12))
sns.heatmap(
    confusion,
    annot=True, fmt="d",
    xticklabels=[MOVEMENTS[m].label_es for m in MOVEMENT_NAMES],
    yticklabels=[MOVEMENTS[m].label_es for m in MOVEMENT_NAMES],
    cmap="Blues"
)
plt.xlabel("Predicho")
plt.ylabel("Real")
plt.title("Matriz de Confusión (Test)")
plt.tight_layout()
plt.savefig("checkpoints/confusion_matrix.png", dpi=150)
plt.show()

## 6. Exportar Modelo Final

In [ ]:
export_path = trainer.export_model(output_dir)
print(f"Modelo exportado: {export_path}")
print(f"Tamaño: {export_path.stat().st_size / 1024:.1f} KB")

## 7. Prueba de Inferencia

In [ ]:
from src.inference import OrquestadorInference

infer = OrquestadorInference(export_path, smoothing_window=5)

# Probar con cada movimiento
from src.data_generator import (
    generate_imu_features_for_movement,
    generate_emg_features_for_movement,
    generate_ecg_features_for_movement,
    generate_context_features_for_movement
)

print("
Pruebas de inferencia:")
print("-" * 60)
for movement_name in MOVEMENT_NAMES:
    imu = generate_imu_features_for_movement(movement_name, 1)
    emg = generate_emg_features_for_movement(movement_name, 1)
    ecg = generate_ecg_features_for_movement(movement_name, 1)
    ctx = generate_context_features_for_movement(movement_name, imu, 1)
    
    result = infer.predict(imu, emg, ecg, ctx)
    
    status = "OK" if result["movement_name"] == movement_name else "FAIL"
    print(f"  [{status:>4}] Entrada: {movement_name:<16} -> {result["movement_label_es"]:>16} ({result["confidence"]:.1%})")